<a href="https://colab.research.google.com/github/pushpendra-saini-pks/CP_LAB/blob/main/CP_LAB2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CUDA Kernel for 1D array addition

In [ ]:
from numba import cuda
import numpy as np
# -------------------------------
# CUDA Kernel
# -------------------------------
@cuda.jit
def add_arrays(a, b, c):
    i = cuda.grid(1)
    if i < a.size:
      c[i] = a[i] + b[i]
# Global thread index
n = 1024
# Create input arrays on CPU
a = np.arange(n, dtype=np.float32)
b = np.arange(n, dtype=np.float32)
# Allocate output array
c = np.zeros(n, dtype=np.float32)
# Copy arrays to GPU
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.to_device(c)
# CUDA configuration
threads_per_block = 256
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block
# Launch kernel
add_arrays[blocks_per_grid, threads_per_block](d_a, d_b, d_c)
# Copy result back to CPU
c = d_c.copy_to_host()
# Print result
print("First 10 results:", c[:10])

First 10 results: [ 0.  2.  4.  6.  8. 10. 12. 14. 16. 18.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


## CUDA Kernel for 2D array addition

In [ ]:
from numba import cuda
import numpy as np

@cuda.jit
def add_arrays_2d(a, b, c):
    # Get the 2D global thread index
    x, y = cuda.grid(2)

    # Get the shape of the arrays
    rows, cols = a.shape

    # Check bounds to prevent out-of-bounds memory access
    if x < rows and y < cols:
        c[x, y] = a[x, y] + b[x, y]

# -------------------------------
# Host Code
# -------------------------------

# Define dimensions for the 2D arrays
rows = 32
cols = 32

# Create input 2D arrays on CPU
a_2d = np.arange(rows * cols, dtype=np.float32).reshape(rows, cols)
b_2d = np.arange(rows * cols, dtype=np.float32).reshape(rows, cols)

# Allocate output 2D array on CPU
c_2d = np.zeros((rows, cols), dtype=np.float32)

# Copy arrays to GPU
d_a_2d = cuda.to_device(a_2d)
d_b_2d = cuda.to_device(b_2d)
d_c_2d = cuda.to_device(c_2d)

# CUDA configuration for 2D grid and blocks
# Threads per block (e.g., 16x16 threads)
threads_per_block_2d = (16, 16)

# Blocks per grid
blocks_per_grid_x = (rows + threads_per_block_2d[0] - 1) // threads_per_block_2d[0]
blocks_per_grid_y = (cols + threads_per_block_2d[1] - 1) // threads_per_block_2d[1]
blocks_per_grid_2d = (blocks_per_grid_x, blocks_per_grid_y)

# Launch kernel
add_arrays_2d[blocks_per_grid_2d, threads_per_block_2d](d_a_2d, d_b_2d, d_c_2d)

# Copy result back to CPU
c_2d_host = d_c_2d.copy_to_host()

# Print result (e.g., first few rows and columns)
print("First 5x5 results of 2D addition:")
print(c_2d_host[:5, :5])


First 5x5 results of 2D addition:
[[  0.   2.   4.   6.   8.]
 [ 64.  66.  68.  70.  72.]
 [128. 130. 132. 134. 136.]
 [192. 194. 196. 198. 200.]
 [256. 258. 260. 262. 264.]]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


## CUDA Kernel for Element wise addition

In [ ]:
from numba import cuda
import numpy as np

@cuda.jit
def add_matrices(a, b, c, n, m):
    i, j = cuda.grid(2)
    if i < n and j < m:
        c[i, j] = a[i, j] * b[i, j]

def main():
    n = 6
    m = 6
    a = np.arange(n * m, dtype=np.int32).reshape((n, m))
    b = np.arange(n * m, dtype=np.int32).reshape((n, m))
    c = np.zeros((n, m), dtype=np.int32)
    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_c = cuda.to_device(c)
    threads_per_block = (6, 6)
    blocks_per_grid = ((n + threads_per_block[0] - 1),
                       (m + threads_per_block[1] - 1))
    add_matrices[blocks_per_grid, threads_per_block](d_a, d_b, d_c, n, m)
    cuda.synchronize()
    c = d_c.copy_to_host()
    print("First 10 rows and columns of the result:")
    print(c[:10, :10])

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 121 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


First 10 rows and columns of the result:
[[   0    1    4    9   16   25]
 [  36   49   64   81  100  121]
 [ 144  169  196  225  256  289]
 [ 324  361  400  441  484  529]
 [ 576  625  676  729  784  841]
 [ 900  961 1024 1089 1156 1225]]
